# XPLT Explorer — Catheter/Urethra Simulation

Extracts mesh data and contact pressure from `sample.xplt` + `sample.feb`.

In [ ]:
import struct
import base64
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path

FEB_FILE  = Path('sample.feb')
XPLT_FILE = Path('sample.xplt')
VTP_DIR   = Path('vtp_output')   # output folder for VTP series
VTP_DIR.mkdir(exist_ok=True)

## 1. Parse .feb — Material and Domain Mapping

In [ ]:
tree = ET.parse(FEB_FILE)
root = tree.getroot()

mat_id_to_name = {}
mat_name_to_id = {}
for mat in root.findall('./Material/material'):
    mid  = int(mat.get('id'))
    name = mat.get('name')
    mat_id_to_name[mid]  = name
    mat_name_to_id[name] = mid

print('Materials:')
for mid, name in sorted(mat_id_to_name.items()):
    print(f'  id={mid}  name="{name}"')

In [ ]:
part_to_mat = {d.get('name'): d.get('mat')
               for d in root.findall('./MeshDomains/SolidDomain')}
print('Parts -> Materials:')
for part, mat in part_to_mat.items():
    print(f'  {part} -> {mat}')

## 2. Parse .xplt Binary — Custom Reader

FEBio xplt v53: top-level blocks `[ROOT, MESH, STATE×180]`.  
Each block: `[tag:4B, size:4B, data:sizeB]` (little-endian).

In [ ]:
def iter_blocks(data: bytes):
    pos = 0
    while pos + 8 <= len(data):
        tag  = struct.unpack_from('<I', data, pos)[0]
        size = struct.unpack_from('<I', data, pos + 4)[0]
        pos += 8
        if pos + size > len(data):
            break
        yield tag, size, data[pos:pos + size]
        pos += size

def find_block(data: bytes, target_tag: int):
    for tag, size, chunk in iter_blocks(data):
        if tag == target_tag:
            return chunk
    return None

TAG = dict(
    ROOT           = 0x01000000,
    MESH           = 0x01040000,
    NODE_SECTION   = 0x01041000,
    NODE_COORDS    = 0x01041200,
    DOMAIN_SECTION = 0x01042000,
    DOMAIN         = 0x01042100,
    DOMAIN_HEADER  = 0x01042101,
    DOM_ELEM_TYPE  = 0x01042102,
    DOM_PART_ID    = 0x01042103,
    DOM_N_ELEMS    = 0x01032104,
    DOM_NAME       = 0x01032105,
    DOM_ELEM_LIST  = 0x01042200,
    ELEMENT        = 0x01042201,
    SURF_SECTION   = 0x01043000,
    SURFACE        = 0x01043100,
    SURF_HEADER    = 0x01043101,
    SURF_ID        = 0x01043102,
    SURF_N_FACETS  = 0x01043103,
    SURF_NAME      = 0x01043104,
    FACET_LIST     = 0x01043200,
    FACET          = 0x01043201,
    STATE          = 0x02000000,
    STATE_HEADER   = 0x02010000,
    STATE_TIME     = 0x02010002,
    STATE_DATA     = 0x02020000,
    STATE_VARIABLE = 0x02020001,
    STATE_VAR_ID   = 0x02020002,
    STATE_VAR_DATA = 0x02020003,
    SURFACE_DATA   = 0x02020500,
)
print('Tag constants loaded.')

In [ ]:
with open(XPLT_FILE, 'rb') as f:
    raw = f.read()

print(f'File size: {len(raw)/1e6:.1f} MB')

top_blocks      = {}
state_positions = []
pos = 4
while pos + 8 < len(raw):
    tag  = struct.unpack_from('<I', raw, pos)[0]
    size = struct.unpack_from('<I', raw, pos + 4)[0]
    if tag == TAG['STATE']:
        state_positions.append((pos, size))
    else:
        top_blocks[tag] = (pos, size)
    pos += 8 + size

print(f'Top-level blocks : {[hex(k) for k in top_blocks]}')
print(f'State blocks     : {len(state_positions)}')

mesh_pos, mesh_size = top_blocks[TAG['MESH']]
mesh_data = raw[mesh_pos + 8 : mesh_pos + 8 + mesh_size]

## 3. Node Coordinates

In [ ]:
node_section    = find_block(mesh_data, TAG['NODE_SECTION'])
node_coords_raw = find_block(node_section, TAG['NODE_COORDS'])

# Each node: [global_id (uint32 stored as float32 bytes), x, y, z]
node_arr        = np.frombuffer(node_coords_raw, dtype='<f4').reshape(-1, 4)
node_ids_global = np.frombuffer(node_arr[:, 0].tobytes(), dtype='<u4')
coords          = node_arr[:, 1:4].astype(np.float64)   # shape (N, 3)

print(f'Nodes : {len(coords)}')
print(f'X     : [{coords[:,0].min():.2f}, {coords[:,0].max():.2f}]')
print(f'Y     : [{coords[:,1].min():.2f}, {coords[:,1].max():.2f}]')
print(f'Z     : [{coords[:,2].min():.2f}, {coords[:,2].max():.2f}]')

## 4. Element Domains

In [ ]:
domain_section = find_block(mesh_data, TAG['DOMAIN_SECTION'])
domains = []

for tag, size, chunk in iter_blocks(domain_section):
    if tag != TAG['DOMAIN']:
        continue
    dom = {}
    for t2, s2, c2 in iter_blocks(chunk):
        if t2 == TAG['DOMAIN_HEADER']:
            for t3, s3, c3 in iter_blocks(c2):
                if   t3 == TAG['DOM_ELEM_TYPE']: dom['elem_type'] = struct.unpack_from('<I', c3)[0]
                elif t3 == TAG['DOM_PART_ID']:   dom['part_id']   = struct.unpack_from('<I', c3)[0]
                elif t3 == TAG['DOM_N_ELEMS']:   dom['n_elems']   = struct.unpack_from('<I', c3)[0]
                elif t3 == TAG['DOM_NAME']:      dom['name']      = c3.rstrip(b'\x00').decode('latin-1').strip()
        elif t2 == TAG['DOM_ELEM_LIST']:
            elems = []
            for t3, s3, c3 in iter_blocks(c2):
                if t3 == TAG['ELEMENT']:
                    elems.append(struct.unpack_from(f'<{s3//4}I', c3))
            dom['elements'] = np.array(elems, dtype=np.int32)
    dom['mat_name'] = mat_id_to_name.get(dom.get('part_id'), 'unknown')
    domains.append(dom)

for d in domains:
    print(f'  part_id={d["part_id"]}  mat="{d["mat_name"]}"  n_elems={d["n_elems"]}')

## 5. Catheter Mesh Summary

In [ ]:
CATHETER_MATS = {'catheter_body', 'catheter_tip'}
cath_domains  = [d for d in domains if d['mat_name'] in CATHETER_MATS]

for d in cath_domains:
    node_idx = np.unique(d['elements'][:, 1:])
    c = coords[node_idx]
    print(f'{d["mat_name"]}: {d["n_elems"]} elements, {len(node_idx)} nodes, '
          f'Z=[{c[:,2].min():.1f}, {c[:,2].max():.1f}] mm')

tip_domain   = next(d for d in domains if d['mat_name'] == 'catheter_tip')
tip_node_idx = np.unique(tip_domain['elements'][:, 1:].ravel())
TIP_Z_MIN    = coords[tip_node_idx, 2].min()
TIP_Z_MAX    = coords[tip_node_idx, 2].max()
print(f'\ncatheter_tip Z range: [{TIP_Z_MIN:.2f}, {TIP_Z_MAX:.2f}] mm')

## 6. Surface Facets

In [ ]:
surf_section  = find_block(mesh_data, TAG['SURF_SECTION'])
xplt_surfaces = {}

for tag, size, chunk in iter_blocks(surf_section):
    if tag != TAG['SURFACE']:
        continue
    surf = {}
    for t2, s2, c2 in iter_blocks(chunk):
        if t2 == TAG['SURF_HEADER']:
            for t3, s3, c3 in iter_blocks(c2):
                if   t3 == TAG['SURF_ID']:       surf['id']       = struct.unpack_from('<I', c3)[0]
                elif t3 == TAG['SURF_N_FACETS']: surf['n_facets'] = struct.unpack_from('<I', c3)[0]
                elif t3 == TAG['SURF_NAME']:     surf['name']     = c3.rstrip(b'\x00').decode('latin-1').strip()
        elif t2 == TAG['FACET_LIST']:
            facets = []
            for t3, s3, c3 in iter_blocks(c2):
                if t3 == TAG['FACET']:
                    v = struct.unpack_from('<5I', c3)
                    facets.append(v[2:5])          # (n0, n1, n2) — 0-indexed into coords
            surf['facets'] = np.array(facets, dtype=np.int32)
    xplt_surfaces[surf['id']] = surf

print(f'Surfaces: {len(xplt_surfaces)}')
for sid, s in sorted(xplt_surfaces.items()):
    print(f'  id={sid:2d}  n={s["n_facets"]:6d}  {s["name"]}')

## 7. Facet Geometry — Centroids and Areas

In [ ]:
# catheter_slidePrimary = surface id 3
facets    = xplt_surfaces[3]['facets']          # shape (23734, 3)
N_FACETS  = len(facets)
f_coords  = coords[facets]                      # shape (N, 3, 3)  [facet, vertex, xyz]

# Centroids
centroids = f_coords.mean(axis=1)               # shape (N, 3)

# Areas: 0.5 * |AB × AC|
AB    = f_coords[:, 1, :] - f_coords[:, 0, :]  # shape (N, 3)
AC    = f_coords[:, 2, :] - f_coords[:, 0, :]  # shape (N, 3)
cross = np.cross(AB, AC)                        # shape (N, 3)
areas = 0.5 * np.linalg.norm(cross, axis=1)    # shape (N,) in mm²

# Tip-zone mask
tip_mask = (centroids[:, 2] >= TIP_Z_MIN) & (centroids[:, 2] <= TIP_Z_MAX)

print(f'catheter_slidePrimary facets : {N_FACETS}')
print(f'Total surface area           : {areas.sum():.2f} mm²')
print(f'Facet area range             : [{areas.min():.4f}, {areas.max():.4f}] mm²')
print(f'Centroid Z range             : [{centroids[:,2].min():.2f}, {centroids[:,2].max():.2f}] mm')
print(f'Tip-zone facets (Z in catheter_tip range): {tip_mask.sum()}')

## 8. Parse All Timesteps — Contact Pressure Matrix

In [ ]:
# Surface data binary layout (xplt v53):
#   For each contact surface: [surf_id (int→f32), byte_count (int→f32), f0…fn]
#   Contact order in file: catheter_slide, cath_body_tip, z1z2, z2z3, z3z4
CONTACT_ORDER = [(3,4),(5,6),(7,8),(9,12),(13,10)]

surface_data_offset = {}
p = 0
for pid, sid in CONTACT_ORDER:
    for surf_id in (pid, sid):
        p += 2                            # skip [id, byte_count] header
        surface_data_offset[surf_id] = p
        p += xplt_surfaces[surf_id]['n_facets']

CP_START = surface_data_offset[3]         # offset of catheter_slidePrimary
CP_END   = CP_START + N_FACETS
print(f'catheter_slidePrimary data at flat indices [{CP_START}, {CP_END})')

In [ ]:
VAR_CONTACT_PRESSURE = 1
VAR_FACET_AREA       = 4

def extract_surface_variable(state_chunk: bytes, var_id: int) -> np.ndarray:
    sd  = find_block(state_chunk, TAG['STATE_DATA'])
    svd = find_block(sd, TAG['SURFACE_DATA'])
    if svd is None: return None
    for tag, size, chunk in iter_blocks(svd):
        if tag != TAG['STATE_VARIABLE']: continue
        vid = arr = None
        for t2, s2, c2 in iter_blocks(chunk):
            if   t2 == TAG['STATE_VAR_ID']:   vid = struct.unpack_from('<I', c2)[0]
            elif t2 == TAG['STATE_VAR_DATA']: arr = np.frombuffer(c2, dtype='<f4')
        if vid == var_id: return arr
    return None

def extract_timestep(state_chunk: bytes) -> float:
    h = find_block(state_chunk, TAG['STATE_HEADER'])
    return float(struct.unpack_from('<f', find_block(h, TAG['STATE_TIME']))[0])

print('Helper functions defined.')

In [ ]:
# Parse all 180 states — ~30-60 s
timesteps        = []
contact_pressure = []

for i, (spos, ssize) in enumerate(state_positions):
    sc  = raw[spos + 8 : spos + 8 + ssize]
    t   = extract_timestep(sc)
    flat = extract_surface_variable(sc, VAR_CONTACT_PRESSURE)
    cp  = flat[CP_START:CP_END].copy() if flat is not None else np.zeros(N_FACETS, dtype='f4')
    timesteps.append(t)
    contact_pressure.append(cp)
    if i % 30 == 0:
        print(f'  state {i+1:3d}/{len(state_positions)}, t={t:.3f}')

timesteps = np.array(timesteps)
cp_matrix = np.stack(contact_pressure)   # shape (n_timesteps, n_facets)

print(f'\ncp_matrix shape : {cp_matrix.shape}')
print(f'Time range      : {timesteps[0]:.3f} → {timesteps[-1]:.3f} s')
print(f'Max cp overall  : {cp_matrix.max():.6f} MPa')

## 9. Full Facet DataFrame

One row per facet: `face_id`, centroid `(cx, cy, cz)`, `area_mm2`, and contact pressure at every timestep.

In [ ]:
df_facets = pd.DataFrame({
    'face_id'  : np.arange(1, N_FACETS + 1),   # 1-indexed to match FEBio convention
    'cx_mm'    : centroids[:, 0],
    'cy_mm'    : centroids[:, 1],
    'cz_mm'    : centroids[:, 2],
    'area_mm2' : areas,
    'in_tip_zone': tip_mask,
})

# Add one column per timestep
for i, t in enumerate(timesteps):
    df_facets[f'cp_t{t:.4f}'] = cp_matrix[i]

print(f'DataFrame shape : {df_facets.shape}  ({N_FACETS} rows × {len(df_facets.columns)} cols)')
print('\nFirst 5 rows (static columns only):')
print(df_facets[['face_id','cx_mm','cy_mm','cz_mm','area_mm2','in_tip_zone']].head())

In [ ]:
# Top facets by peak contact pressure (max over all timesteps)
cp_cols = [c for c in df_facets.columns if c.startswith('cp_t')]
df_facets['cp_peak'] = df_facets[cp_cols].max(axis=1)

print('Top 15 facets by peak contact pressure:')
print(df_facets.nlargest(15, 'cp_peak')
      [['face_id','cx_mm','cy_mm','cz_mm','area_mm2','in_tip_zone','cp_peak']]
      .to_string(index=False))

## 10. Contact Pressure vs Time Plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# --- Aggregate statistics over time ---
cp_max       = cp_matrix.max(axis=1)
cp_mean_nz   = np.array([row[row > 0].mean() if (row > 0).any() else 0.0
                          for row in cp_matrix])
n_contact    = (cp_matrix > 0).sum(axis=1)          # number of facets in contact
cp_tip_max   = cp_matrix[:, tip_mask].max(axis=1)
cp_tip_mean  = np.array([row[row > 0].mean() if (row > 0).any() else 0.0
                          for row in cp_matrix[:, tip_mask]])

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('catheter_slidePrimary — Contact Pressure vs Time', fontsize=13)

# Top-left: max & mean contact pressure
ax = axes[0, 0]
ax.plot(timesteps, cp_max,     label='Max  (all facets)',  color='crimson')
ax.plot(timesteps, cp_mean_nz, label='Mean (in-contact)',  color='steelblue', linestyle='--')
ax.plot(timesteps, cp_tip_max, label='Max  (tip zone)',    color='darkorange', linestyle=':')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Contact Pressure (MPa)')
ax.set_title('Max / Mean Contact Pressure')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.4)

# Top-right: number of facets in contact
ax = axes[0, 1]
ax.fill_between(timesteps, n_contact, alpha=0.4, color='teal')
ax.plot(timesteps, n_contact, color='teal')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Count')
ax.set_title('Number of Facets in Contact (cp > 0)')
ax.grid(True, alpha=0.4)

# Bottom-left: facet centroids coloured by peak cp (all facets)
ax = axes[1, 0]
sc = ax.scatter(df_facets.cz_mm, df_facets.cy_mm,
                c=df_facets.cp_peak, cmap='hot_r', s=1,
                vmin=0, vmax=np.percentile(df_facets.cp_peak, 99))
plt.colorbar(sc, ax=ax, label='Peak cp (MPa)')
# highlight tip zone boundary
ax.axvline(TIP_Z_MIN, color='cyan', linewidth=0.8, linestyle='--', label='tip zone')
ax.axvline(TIP_Z_MAX, color='cyan', linewidth=0.8, linestyle='--')
ax.set_xlabel('Z (mm)')
ax.set_ylabel('Y (mm)')
ax.set_title('Facet Map — Peak Contact Pressure')
ax.legend(fontsize=8)

# Bottom-right: tip-zone facets, cp time evolution of top-10 highest-peak facets
ax = axes[1, 1]
top10_ids = df_facets[df_facets.in_tip_zone].nlargest(10, 'cp_peak')['face_id'].values
cmap_lines = cm.get_cmap('tab10', 10)
for k, fid in enumerate(top10_ids):
    row = df_facets[df_facets.face_id == fid].iloc[0]
    cp_trace = np.array([row[f'cp_t{t:.4f}'] for t in timesteps])
    ax.plot(timesteps, cp_trace, color=cmap_lines(k), linewidth=0.8,
            label=f'f{fid} (z={row.cz_mm:.1f})')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Contact Pressure (MPa)')
ax.set_title('Top-10 Tip Facets — cp vs Time')
ax.legend(fontsize=6, ncol=2)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('contact_pressure_analysis.png', dpi=150)
plt.show()
print('Saved: contact_pressure_analysis.png')

## 11. Export VTP Series + PVD

Writes one `.vtp` file per timestep into `vtp_output/` plus a `catheter_slide_primary.pvd` index.  
Open the `.pvd` in **ParaView** to animate contact pressure over time.

Each VTP contains:
- **Points**: unique node coordinates used by `catheter_slidePrimary` (undeformed mesh)
- **Polys**: tri3 connectivity (re-indexed to local point IDs)
- **CellData**: `face_id`, `area_mm2`, `contact_pressure_MPa`

In [ ]:
# --- Build local node mapping for this surface ---
# facets contains global node indices (0-based into coords)
unique_global_idx, inv = np.unique(facets.ravel(), return_inverse=True)
local_facets  = inv.reshape(-1, 3)                          # shape (N_FACETS, 3)
local_coords  = coords[unique_global_idx].astype(np.float32)# shape (N_pts, 3)
N_PTS         = len(local_coords)

# VTK connectivity / offsets arrays
connectivity  = local_facets.ravel().astype(np.int32)       # shape (N_FACETS*3,)
offsets       = np.arange(3, N_FACETS * 3 + 1, 3, dtype=np.int32)  # [3,6,9,...]

face_ids_arr  = np.arange(1, N_FACETS + 1, dtype=np.int32)
areas_f32     = areas.astype(np.float32)

print(f'Local mesh: {N_PTS} pts, {N_FACETS} triangles')


def _b64(arr: np.ndarray) -> str:
    """Encode numpy array as VTK binary base64: [uint32 nbytes][data bytes]."""
    data  = arr.tobytes()
    hdr   = struct.pack('<I', len(data))
    return base64.b64encode(hdr + data).decode('ascii')


def write_vtp(filepath: Path, cp_arr: np.ndarray, timestep: float):
    """Write a single VTP file for one timestep."""
    lines = [
        '<?xml version="1.0"?>',
        '<VTKFile type="PolyData" version="0.1" byte_order="LittleEndian" header_type="UInt32">',
        '  <PolyData>',
        f'  <Piece NumberOfPoints="{N_PTS}" NumberOfVerts="0" NumberOfLines="0"',
        f'         NumberOfStrips="0" NumberOfPolys="{N_FACETS}">',

        # ---- Points ----
        '    <Points>',
        '      <DataArray type="Float32" NumberOfComponents="3" format="binary">',
        f'        {_b64(local_coords)}',
        '      </DataArray>',
        '    </Points>',

        # ---- Polys ----
        '    <Polys>',
        '      <DataArray type="Int32" Name="connectivity" format="binary">',
        f'        {_b64(connectivity)}',
        '      </DataArray>',
        '      <DataArray type="Int32" Name="offsets" format="binary">',
        f'        {_b64(offsets)}',
        '      </DataArray>',
        '    </Polys>',

        # ---- Cell data ----
        '    <CellData Scalars="contact_pressure_MPa">',
        '      <DataArray type="Int32" Name="face_id" format="binary">',
        f'        {_b64(face_ids_arr)}',
        '      </DataArray>',
        '      <DataArray type="Float32" Name="area_mm2" format="binary">',
        f'        {_b64(areas_f32)}',
        '      </DataArray>',
        '      <DataArray type="Float32" Name="contact_pressure_MPa" format="binary">',
        f'        {_b64(cp_arr.astype(np.float32))}',
        '      </DataArray>',
        '    </CellData>',

        '  </Piece>',
        '  </PolyData>',
        '</VTKFile>',
    ]
    filepath.write_text('\n'.join(lines))


print('VTP writer ready.')

In [ ]:
# Write all VTP files + PVD index
vtp_filenames = []

for i, (t, cp_row) in enumerate(zip(timesteps, cp_matrix)):
    fname = f'catheter_slide_t{i:04d}.vtp'
    fpath = VTP_DIR / fname
    write_vtp(fpath, cp_row, t)
    vtp_filenames.append((t, fname))
    if i % 30 == 0:
        print(f'  Written {i+1}/{len(timesteps)}: {fname}  (t={t:.3f})')

# PVD collection file
pvd_lines = [
    '<?xml version="1.0"?>',
    '<VTKFile type="Collection" version="0.1" byte_order="LittleEndian">',
    '  <Collection>',
]
for t, fname in vtp_filenames:
    pvd_lines.append(f'    <DataSet timestep="{t:.6f}" group="" part="0" file="{fname}"/>')
pvd_lines += ['  </Collection>', '</VTKFile>']

pvd_path = VTP_DIR / 'catheter_slide_primary.pvd'
pvd_path.write_text('\n'.join(pvd_lines))

print(f'\nDone. {len(vtp_filenames)} VTP files + PVD written to "{VTP_DIR}/"')
print(f'Open in ParaView: {pvd_path}')

In [ ]:
# Optional: export the full DataFrame to CSV
# (large file — all facets × all timesteps)
# df_facets.to_csv('catheter_slide_all_facets.csv', index=False)

# Lightweight export: static geometry + peak cp only
df_facets[['face_id','cx_mm','cy_mm','cz_mm','area_mm2','in_tip_zone','cp_peak']].to_csv(
    'catheter_slide_primary_summary.csv', index=False
)
print('Saved: catheter_slide_primary_summary.csv')